In [53]:
import pandas as pd 
import numpy as np 
import os 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import ttest_ind
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
os.makedirs(os.path.join(os.getcwd(),'images'),exist_ok=True)
os.makedirs(os.path.join(os.getcwd(),'datasets'),exist_ok=True)
IMAGE_DIR = os.path.join(os.getcwd(),'images')
DATASETS = os.path.join(os.getcwd(),'datasets','EPILEPSY_worked_example')

# Heatmaps for pre-ictal activity, ictal-activity and relative increase with respect to pre-ictal 

## Event 8 

In [54]:
def compute_hjorth_parameters(arr):
    # Activity: variance along time (axis=1)

    first_derivative = np.diff(arr, n=1, axis=1)
    second_derivative = np.diff(arr, n=2, axis=1)
    
    activity = np.var(arr, axis=1)
    mobility = np.std(first_derivative, axis=1) / np.std(arr, axis=1)
    complexity  = (np.std(second_derivative, axis=1) / np.std(first_derivative, axis=1)) / mobility
    
    features = {
        "Activity" : activity,
        "Mobility" : mobility,
        "Complexity" : complexity,
        
    }
    return pd.DataFrame(features)


In [55]:
def generate_comparison_plots(event):
   """
       Generates 3 plots showing the difference between ict and pre-ict 
       
   """
   
   ict_data = event['ict_data']
   pre_data = event['pre_data']
   baseline_means = event['baseline_means'] #these mean values were gerneated with pre_data 
   baseline_stds = event['baseline_stds'] # same here 
    
   zscore = (ict_data - baseline_means[:, None]) / baseline_stds[:, None]
    
   fig = go.Figure()
   fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=("pre-ictal activity", "ictal activity", "Rel.Increase with respect to pre-ictal",))

   # plot pre_data
   fig.add_trace(
      go.Heatmap(
            z=pre_data,
            x= [i for i in range(1,pre_data.shape[1]+1)],
            y= [i for i in range(1,pre_data.shape[0]+1)],
            colorscale='Viridis',
            colorbar=dict(title='z-score')
      ),
      row=1,
      col=1
   )
   
   # plot ict_data
   fig.add_trace(
      go.Heatmap(
            z=ict_data,
            x= [i for i in range(1,ict_data.shape[1]+1)],
            y= [i for i in range(1,ict_data.shape[0]+1)],
            colorscale='Viridis',
            colorbar=dict(title='z-score')
      ),
      row=1,
      col=2
   )
   # plot zcrore
   fig.add_trace(
      go.Heatmap(
            z=zscore,
            x= [i for i in range(1,zscore.shape[1]+1)],
            y= [i for i in range(1,zscore.shape[0]+1)],
            colorscale='Viridis',
            colorbar=dict(title='z-score')
      ),
      row=1,
      col=3
   )

   fig.update_layout(title_text=f"Event {event['event_number']}",
                     height=1000,
                     width=1600)

   fig.show()


def generate_feature_ordering_line_plot(Z_abs):
    # --- 1. Line plot of t-statistics ---
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(
        x = [i for i in range(1, Z_abs.shape[0]+1)],
        y=Z_abs,
        mode='lines',
        name='t-statistics'
    ))
    fig1.update_layout(
        title='Feature Ordering ',
        xaxis_title='Feature no',
        yaxis_title='Score'
    )
    fig1.show()

def generate_feature_ordering_heatmap(num_sensors, Z_abs):

# --- 2. Heatmap (3 attributes × sensors) ---
    Z_matrix = Z_abs.reshape((3, num_sensors), order='F')

    attribute_names = ["Activity", "Mobility", "Complexity"]

    num_sensors = Z_matrix.shape[1]

    fig2 = go.Figure(data=go.Heatmap(
        z=Z_matrix,
        x=list(range(1, num_sensors+1)),      # sensors 1..76
        y=attribute_names[::-1],              # reverse so Activity is top if needed
        colorbar=dict(title=""),
    ))

    # Correct tick marks: 5,10,15,... (convert to 0-based positions)
    sensor_ticks = list(range(5, num_sensors+1, 5))
    sensor_labels = [str(s) for s in sensor_ticks]

    fig2.update_xaxes(
        tickmode='array',
        tickvals=sensor_ticks,
        ticktext=sensor_labels,
        title_text="Sensor no"
    )

    fig2.update_yaxes(
        tickmode='array',
        tickvals=[0,1,2],
        ticktext=attribute_names,
        title_text="Attribute"
    )

    fig2.update_layout(
        title="Feature Ordering (Heatmap) — MATLAB-like",
        autosize=True
    )

    fig2.show()



In [56]:
def decode_feature_matlab(idx):
    attribute_names = ["Activity", "Mobility", "Complexity"]
    sensor = (idx // 3) + 1
    r = idx % 3
    attribute = 3 if r == 0 else r   
    attribute_name = attribute_names[attribute - 1]  
    return sensor, attribute_name

# Generate plots of zscore for all events

In [57]:
events = []
for i in range(1, 9):
    event = {}
    event['event_number'] = i 
    pre_data = np.loadtxt(os.path.join(DATASETS, f"sz{event['event_number']}_pre.dat"))
    ict_data = np.loadtxt(os.path.join(DATASETS, f"sz{event['event_number']}_ict.dat"))
    baseline_means = np.mean(pre_data, axis=1)
    baseline_stds  = np.std(pre_data, axis=1)
    
    event['ict_data'] = ict_data
    event['pre_data'] = pre_data
    event['baseline_means'] = baseline_means
    event['baseline_stds'] = baseline_stds
    
    events.append(event)

In [58]:
df = pd.DataFrame(events)
#df.apply(lambda row: generate_comparison_plots(row),axis=1 )
df['hjorth_parameters_ict'] = df.apply(lambda row: compute_hjorth_parameters(row['ict_data']),axis=1)
df['hjorth_parameters_pre'] = df.apply(lambda row: compute_hjorth_parameters(row['pre_data']),axis=1)

In [59]:
# 76 windows × 3 Hjorth features × 8 channels
ictal_hjorth   = np.stack(df['hjorth_parameters_ict'].values, axis=2)
preictal_hjorth = np.stack(df['hjorth_parameters_pre'].values, axis=2)

# Build feature vectors exactly like MATLAB
# Each column: all 3 features × 8 channels for one window
ictal_feature_matrix   = np.hstack([ictal_hjorth[i, :, :].T   for i in range(76)])  # 8×3 per window → 8×(3*76)
preictal_feature_matrix = np.hstack([preictal_hjorth[i, :, :].T for i in range(76)])  # 8×(3*76)

# Stack ictal (class 1) and preictal (class 0) vertically
all_features = np.vstack([ictal_feature_matrix, preictal_feature_matrix])  # 16 × 228
group_labels = np.array([1] * 8 + [0] * 8)  # 1 = ictal, 0 = preictal

# Compute t-test (vectorized across feature columns)
t_statistic, p_values = ttest_ind(
    all_features[group_labels == 1],
    all_features[group_labels == 0],
    axis=0
)

t_stat_abs = np.abs(t_statistic)

# Rank features by absolute t-statistic (descending)
feature_rank_indices = np.argsort(-t_stat_abs)

generate_feature_ordering_line_plot(t_stat_abs)
generate_feature_ordering_heatmap(76, t_stat_abs)
worst_idx = feature_rank_indices[-1]
worst_feature_index = feature_rank_indices[-1]
worst_sensor, worst_attr = decode_feature_matlab(worst_idx)
print(f"WORST feature: Sensor {worst_sensor}, Attribute {worst_attr}, Score={t_stat_abs[worst_idx]:.4f}")



WORST feature: Sensor 29, Attribute Mobility, Score=0.0541


In [75]:
feature_rank_indices

array([ 28, 220,  52,  49,  34,  55,  10,  31,  53,   3,   4,  24, 219,
       221,  27,  25,   1,   7,   0,  73,  30,  35,  76, 210,   9,  56,
        50, 162,  57,   6,  80,  26,  77,   5,  74,  33,  58,  60, 213,
        32, 105,  81,  29,   2,   8,  36, 222, 100, 147, 165,  54, 101,
        11,  78, 202, 124, 103,  79, 199, 225,  59,  51, 198, 144, 189,
        16, 141, 192,  12, 126, 203, 121,  84, 168, 125,  97, 104,  93,
       102,  66,  42,  87,  48,  69, 159,  95, 214,  63, 185, 204, 122,
        98,  82, 196, 117, 200, 171, 108,  13,  17,  39, 161,  19, 129,
       127, 150, 134, 143, 223,  83,  14,  96, 114, 137,  45, 111,  38,
       208, 138, 197, 195,  99,  22, 119, 110,  37, 215,  90,  72, 128,
        71, 218, 226,  47,  15, 183, 145, 151, 224,  75, 174, 146, 115,
       216, 132, 207, 212, 113, 205, 209,  68,  18,  21, 135,  62, 107,
       186, 154, 116, 153, 179, 156, 164, 177, 191, 180, 167,  20, 227,
       169, 130,  40, 120,  61,  92, 176,  89,  94, 149, 123, 11

In [86]:
#####
###  NOTE: use the worst sensor's features 
#####

worst_features_idx = worst_idx +1
X_worst = all_features[:,worst_features_idx:worst_features_idx+3]  # we get the 3 features of sensor 29
y = group_labels  

lda = LinearDiscriminantAnalysis()
lda.fit(X_worst, y)
y_pred = lda.predict(X_worst)

accuracy = accuracy_score(y, y_pred)
print(f"Classification performance (CorrectRate): {accuracy:.4f}")


Classification performance (CorrectRate): 1.0000


In [91]:
import plotly.express as px
from sklearn.decomposition import PCA
import numpy as np

pca = PCA(n_components=2)
X_pca = pca.fit_transform(all_features)

fig = px.scatter(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    color=group_labels.astype(str),  # convert 0/1 to string for legend
    labels={'color': 'Class'},
    title='Pre-ictal and Ictal classes',
    symbol=group_labels.astype(str),
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')))
fig.show()


In [51]:
worst_features_idx

array([220,  28,  52,  49,  53,  34,  10,  55,  31, 221,   3,  25,   4,
         0, 219,  24,   1,  27,   7,  73,  50, 162,  35,  30,  77,  76,
        26,   9,  74, 210,  56,   5,   6,  57,   8,  32, 222,  80,  33,
       213,   2, 105,  29,  11, 101,  58,  60, 225, 165,  81, 100, 199,
        36,  59,  54,  78, 147, 202, 125, 103, 189, 168,  51, 124,  79,
       198, 159, 104,  16, 144, 141,  42, 200, 185, 216,  84, 121,  12,
       203, 126, 192,  97,  48, 122,  66,  98, 102,  69,  87,  93, 161,
        95,  63, 223,  17, 204,  39, 171, 214, 108, 196,  19,  14, 143,
        13,  83, 134, 129,  38,  82, 150,  45, 117, 197, 137, 110,  22,
       127, 114, 138, 111, 226, 119, 215,  96, 183,  99, 224, 153, 195,
       174,  47,  37, 208, 128, 180, 177,  72, 113, 212,  71, 116, 146,
        90, 164, 132, 186, 107,  18,  68,  15,  75, 227, 145, 156, 151,
       207, 135, 205,  62,  21, 167, 209, 179, 115,  89, 218, 154,  20,
        94, 191, 160,  40,  92, 184, 217, 140, 149, 118, 206, 15

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_worst, y, test_size=0.3, stratify=y
)

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy (CorrectRate): {accuracy:.4f}")


Test accuracy (CorrectRate): 1.0000


In [47]:
y

array([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])

In [46]:
# 1) Split first, on all features
X_train_all, X_test_all, y_train, y_test = train_test_split(
    all_features, y, test_size=0.3, stratify=y, random_state=42
)

# 2) Compute t-test on training only
t_statistic, p_values = ttest_ind(
    X_train_all[y_train == 1],
    X_train_all[y_train == 0],
    axis=0
)
t_stat_abs = np.abs(t_statistic)
feature_rank_indices = np.argsort(-t_stat_abs)

# 3) Select worst features from TRAIN ranking
worst_features_idx = feature_rank_indices[-1:]  # e.g. single worst
X_train_worst = X_train_all[:, worst_features_idx]
X_test_worst  = X_test_all[:,  worst_features_idx]

# 4) Train and test
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_worst, y_train)
y_pred = lda.predict(X_test_worst)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy (CorrectRate): {accuracy:.4f}")


Test accuracy (CorrectRate): 0.4000


Test accuracy (worst features): 1.0000


In [42]:
X_train

array([[0.04348389, 0.04319207, 0.0535366 , ..., 0.08820809, 0.08564686,
        0.11550082],
       [0.30651151, 0.20433255, 0.30649522, ..., 0.12748705, 0.11971903,
        0.17290219],
       [0.04577686, 0.04646308, 0.06104358, ..., 0.15904536, 0.14797847,
        0.23378956],
       ...,
       [0.28853217, 0.20691904, 0.29443599, ..., 0.10079339, 0.09109322,
        0.11511664],
       [0.04171683, 0.03548473, 0.06155476, ..., 0.10818307, 0.08602775,
        0.19237396],
       [0.29533399, 0.20551873, 0.30478064, ..., 0.12364936, 0.1075599 ,
        0.16122126]], shape=(8, 227))